# SwarmFusionDNN — Extended 6-Model Ensemble for Brain Tumor Detection

**Based on:** *Swarm intelligence optimization-based fusion of ConvMixer-enhanced deep neural networks for brain tumor detection* (Asif et al.)

**Extension:** Adds **InceptionV3** and **ResNet50** as two additional base models beyond the paper's original four (MobileNet, DenseNet121, MobileNetV2, Xception).

---

## Pipeline
1. Data Loading & Augmentation
2. Build 6 ConvMixer-enhanced CNN Base Models
3. Train Each Model Independently
4. Generate Softmax Probabilities
5. PSO Optimisation (find optimal per-model weights)
6. Weighted-Average Ensemble Prediction
7. Evaluation (Accuracy, Sensitivity, Specificity, Precision, F1, AUC)
8. Grad-CAM Visualisation

## 0. Configuration

> **Set your dataset path below before running.**

In [ ]:
# ── USER SETTINGS ─────────────────────────────────────────────────────────────
DATA_DIR  = "/kaggle/input/brain-tumor-detection/no"   # Kaggle Path
DATASET   = "BR35H"                   # 'BR35H' | 'Figshare' | 'Bangladesh'
N_CLASSES = 2                         # 2 for BR35H (binary), 3 for others
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 20
LR          = 1e-3
PATIENCE    = 12
ALPHA_DROP  = 0.5

# PSO hyper-parameters (from the paper)
PSO_POP   = 50
PSO_ITER  = 100
PSO_W     = 0.5
PSO_C1    = 1.0
PSO_C2    = 2.0

# 6 models (original 4 + 2 new)
MODEL_NAMES = [
    "MobileNet",
    "DenseNet121",
    "MobileNetV2",
    "Xception",
    "InceptionV3",   # NEW
    "ResNet50",       # NEW
]

import os
os.makedirs("outputs", exist_ok=True)
os.makedirs("model_weights", exist_ok=True)
print("Config ready.")

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import (
    MobileNet, DenseNet121, MobileNetV2, Xception,
    InceptionV3, ResNet50
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, accuracy_score, f1_score,
    precision_score, recall_score
)

print(f"TensorFlow: {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

## 2. Data Pipeline

In [ ]:
train_aug = ImageDataGenerator(
    rescale=1.0/255.0,
    horizontal_flip=True,
    vertical_flip=True,
    shear_range=0.2,
    rotation_range=20,
    validation_split=0.2,
)
test_aug = ImageDataGenerator(rescale=1.0/255.0)

target = (IMG_SIZE, IMG_SIZE)

train_gen = train_aug.flow_from_directory(
    DATA_DIR, target_size=target, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=42
)
val_gen = train_aug.flow_from_directory(
    DATA_DIR, target_size=target, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=42
)

N_CLASSES   = train_gen.num_classes
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f"Classes ({N_CLASSES}): {CLASS_NAMES}")
print(f"Train samples : {train_gen.samples}")
print(f"Val samples   : {val_gen.samples}")

## 3. Build 6 ConvMixer-Enhanced Models

Each model:
1. Pre-trained backbone (ImageNet, no top)
2. **ConvMixer block** — Depthwise Conv → Pointwise Conv → GeLU → BatchNorm
3. GlobalAveragePooling → **Alpha Dropout (0.5)** → Dense softmax head

In [ ]:
def convmixer_block(x, filters):
    """ConvMixer: depthwise + pointwise convolutions with GeLU and BatchNorm."""
    x = layers.DepthwiseConv2D(kernel_size=3, padding='same', use_bias=False)(x)
    x = layers.Conv2D(filters, kernel_size=1, use_bias=False)(x)   # pointwise
    x = layers.Activation('gelu')(x)
    x = layers.BatchNormalization()(x)
    return x


def build_model(base_fn, n_classes, model_name):
    inp      = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    backbone = base_fn(include_top=False, weights='imagenet', input_tensor=inp)
    backbone.trainable = False   # freeze pre-trained weights

    x       = backbone.output
    filters = x.shape[-1]
    x       = convmixer_block(x, filters)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.AlphaDropout(ALPHA_DROP)(x)
    out     = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inp, outputs=out, name=model_name)
    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


MODEL_REGISTRY = {
    'MobileNet'  : MobileNet,
    'DenseNet121': DenseNet121,
    'MobileNetV2': MobileNetV2,
    'Xception'   : Xception,
    'InceptionV3': InceptionV3,   # NEW
    'ResNet50'   : ResNet50,       # NEW
}

models = {}
for name in MODEL_NAMES:
    print(f'Building {name} ...')
    models[name] = build_model(MODEL_REGISTRY[name], N_CLASSES, name)

print('\nParameter counts:')
for name, m in models.items():
    print(f'  {name:<15}: {m.count_params():,}')

## 4. Train All 6 Models

- Optimizer: **Adam** (lr = 1e-3, with ReduceLROnPlateau factor=0.5)
- EarlyStopping patience = 12
- Epochs = 20

> Skip this cell and load saved weights if you already have checkpoints.

In [ ]:
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
        ModelCheckpoint(f'model_weights/{model_name}_best.keras',
                        monitor='val_accuracy', save_best_only=True, verbose=0),
    ]


histories = {}
for name, model in models.items():
    ckpt = f'model_weights/{name}_best.keras'
    if os.path.exists(ckpt):
        print(f'Loading saved weights for {name}')
        model.load_weights(ckpt)
        continue

    print(f'\n{"="*55}\n  Training {name}\n{"="*55}')
    h = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS, callbacks=get_callbacks(name), verbose=1
    )
    histories[name] = h
    loss, acc = model.evaluate(val_gen, verbose=0)
    print(f'  {name} — Val Acc: {acc*100:.2f}%  Val Loss: {loss:.4f}')

print('\nAll models ready.')

### Training Curves

In [ ]:
if histories:
    n = len(histories)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))
    if n == 1: axes = [axes]
    for ax_row, (name, hist) in zip(axes, histories.items()):
        ax_row[0].plot(hist.history['accuracy'], label='Train')
        ax_row[0].plot(hist.history['val_accuracy'], label='Val')
        ax_row[0].set_title(f'{name} — Accuracy'); ax_row[0].legend(); ax_row[0].grid(True)
        ax_row[1].plot(hist.history['loss'], label='Train')
        ax_row[1].plot(hist.history['val_loss'], label='Val')
        ax_row[1].set_title(f'{name} — Loss'); ax_row[1].legend(); ax_row[1].grid(True)
    plt.tight_layout(); plt.savefig('outputs/training_curves.png', dpi=150); plt.show()
else:
    print('All weights loaded from disk — no training performed this session.')

## 5. Generate Softmax Predictions from All 6 Models

In [ ]:
val_gen.reset()
true_labels = val_gen.classes

all_probs = []   # shape: (N_models, N_samples, N_classes)
for name in MODEL_NAMES:
    val_gen.reset()
    print(f'  Predicting with {name} ...')
    p = models[name].predict(val_gen, verbose=0)
    all_probs.append(p)
    acc_i = accuracy_score(true_labels, np.argmax(p, axis=1))
    print(f'    accuracy = {acc_i*100:.2f}%')

all_probs = np.array(all_probs)
print(f'\nProbability tensor shape: {all_probs.shape}')
# (N_models=6, N_samples, N_classes)

## 6. Particle Swarm Optimisation (PSO)

PSO finds optimal weights $w_i \in [0,1]$ for each of the 6 models by **minimising the classifier error rate**:

$$\text{Error Rate} = \frac{\text{Misclassified}}{\text{Total}} \times 100$$

Parameters (from paper): Population=50, Iterations=100, $w=0.5$, $c_1=1$, $c_2=2$

In [ ]:
N_MODELS = len(MODEL_NAMES)

def ensemble_predict_from_weights(weights, probs):
    w = weights / (weights.sum() + 1e-9)
    weighted = np.einsum('m,mnc->nc', w, probs)
    return np.argmax(weighted, axis=1)

def error_rate(weights, probs, true_labels):
    preds = ensemble_predict_from_weights(weights, probs)
    return (np.sum(preds != true_labels) / len(true_labels)) * 100.0


# ── Initialise swarm ──────────────────────────────────────────────────────────
np.random.seed(42)
pos  = np.random.uniform(0.0, 1.0, (PSO_POP, N_MODELS))
vel  = np.zeros_like(pos)

pbest_pos = pos.copy()
pbest_err = np.array([error_rate(p, all_probs, true_labels) for p in pos])

gbest_idx = np.argmin(pbest_err)
gbest_pos = pbest_pos[gbest_idx].copy()
gbest_err = pbest_err[gbest_idx]

error_history = [gbest_err]
print(f'Initial best error rate: {gbest_err:.2f}%')

# ── Main PSO loop ─────────────────────────────────────────────────────────────
for it in range(PSO_ITER):
    r1 = np.random.rand(PSO_POP, N_MODELS)
    r2 = np.random.rand(PSO_POP, N_MODELS)

    vel = PSO_W * vel + PSO_C1 * r1 * (pbest_pos - pos) + PSO_C2 * r2 * (gbest_pos - pos)
    pos = np.clip(pos + vel, 0.0, 1.0)

    errs = np.array([error_rate(p, all_probs, true_labels) for p in pos])

    improved = errs < pbest_err
    pbest_pos[improved] = pos[improved]
    pbest_err[improved] = errs[improved]

    best_i = np.argmin(pbest_err)
    if pbest_err[best_i] < gbest_err:
        gbest_pos = pbest_pos[best_i].copy()
        gbest_err  = pbest_err[best_i]

    error_history.append(gbest_err)
    if (it + 1) % 10 == 0:
        print(f'  Iter {it+1:>3}/{PSO_ITER}  Error: {gbest_err:.4f}%  Accuracy: {100-gbest_err:.4f}%')

best_weights = gbest_pos
print(f'\nFinal error rate  : {gbest_err:.4f}%')
print(f'Final accuracy    : {100-gbest_err:.4f}%')

norm_w = best_weights / best_weights.sum()
print('\nOptimal weights (normalised):')
for n, w in zip(MODEL_NAMES, norm_w):
    print(f'  {n:<15}: {w:.4f}')

### PSO Convergence Plot

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(error_history, color='royalblue', linewidth=2)
plt.xlabel('Iteration'); plt.ylabel('Error Rate (%)')
plt.title('PSO Convergence — Classifier Error Rate (6 Models)')
plt.grid(True); plt.tight_layout()
plt.savefig('outputs/pso_convergence.png', dpi=150); plt.show()

### PSO Weights per Model

In [ ]:
plt.figure(figsize=(9, 4))
bars = plt.bar(MODEL_NAMES, norm_w, color=plt.cm.tab10.colors[:N_MODELS], edgecolor='white')
for bar, wv in zip(bars, norm_w):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{wv:.3f}', ha='center', va='bottom', fontsize=9)
plt.ylabel('Normalised PSO Weight'); plt.ylim(0, max(norm_w) * 1.25)
plt.title('PSO-Optimised Weights (6 Models)')
plt.grid(axis='y', linestyle='--', alpha=0.5); plt.tight_layout()
plt.savefig('outputs/pso_weights.png', dpi=150); plt.show()

## 7. Evaluation — SwarmFusionDNN Ensemble

In [ ]:
pred_labels = ensemble_predict_from_weights(best_weights, all_probs)

acc  = accuracy_score(true_labels, pred_labels) * 100
f1   = f1_score(true_labels, pred_labels, average='macro') * 100
avg  = 'binary' if N_CLASSES == 2 else 'macro'
prec = precision_score(true_labels, pred_labels, average=avg, zero_division=0) * 100
rec  = recall_score(true_labels, pred_labels, average=avg, zero_division=0) * 100

cm_mat = confusion_matrix(true_labels, pred_labels)
if N_CLASSES == 2:
    tn, fp, fn, tp = cm_mat.ravel()
    spec = tn / (tn + fp) * 100
else:
    spec_list = []
    for i in range(N_CLASSES):
        tn_i = cm_mat.sum() - cm_mat[i,:].sum() - cm_mat[:,i].sum() + cm_mat[i,i]
        fp_i = cm_mat[:,i].sum() - cm_mat[i,i]
        spec_list.append(tn_i / (tn_i + fp_i) if (tn_i + fp_i) > 0 else 0)
    spec = np.mean(spec_list) * 100

# AUC
ens_probs = np.einsum('m,mnc->nc', norm_w, all_probs)
try:
    if N_CLASSES == 2:
        auc = roc_auc_score(true_labels, ens_probs[:,1])
    else:
        auc = roc_auc_score(true_labels, ens_probs, multi_class='ovr', average='macro')
except Exception:
    auc = float('nan')

print('='*55)
print('  SwarmFusionDNN (6 Models) — Final Results')
print('='*55)
print(f'  Accuracy    : {acc:.4f}%')
print(f'  Sensitivity : {rec:.4f}%')
print(f'  Specificity : {spec:.4f}%')
print(f'  Precision   : {prec:.4f}%')
print(f'  F1-Score    : {f1:.4f}%')
print(f'  AUC         : {auc:.4f}')
print(f'  Error Rate  : {100-acc:.4f}%')
print('='*55)

print('\n  Individual model accuracies:')
for i, name in enumerate(MODEL_NAMES):
    m_pred = np.argmax(all_probs[i], axis=1)
    m_acc  = accuracy_score(true_labels, m_pred) * 100
    print(f'    {name:<15}: {m_acc:.2f}%')

print('\n  Classification Report:')
print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES))

### Confusion Matrix

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(cm_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.title('SwarmFusionDNN (6 Models) — Confusion Matrix')
plt.tight_layout(); plt.savefig('outputs/confusion_matrix.png', dpi=150); plt.show()

### Individual Models vs Ensemble Accuracy

In [ ]:
names_chart = MODEL_NAMES + ['SwarmFusionDNN\n(PSO Ensemble)']
accs_chart  = [accuracy_score(true_labels, np.argmax(all_probs[i], axis=1))*100
               for i in range(N_MODELS)] + [acc]

colors = ['#4C72B0'] * N_MODELS + ['#DD8452']
plt.figure(figsize=(12, 5))
bars = plt.bar(names_chart, accs_chart, color=colors, edgecolor='white', width=0.6)
plt.ylabel('Accuracy (%)'); plt.ylim(max(0, min(accs_chart)-5), 101)
plt.title('Individual Models vs. SwarmFusionDNN Ensemble (6 Models)')
for bar, a in zip(bars, accs_chart):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{a:.2f}%', ha='center', va='bottom', fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=20, ha='right'); plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150); plt.show()

### Save Results

In [ ]:
results = {
    'Dataset': DATASET, 'N_Models': N_MODELS,
    'Accuracy (%)': acc, 'Sensitivity (%)': rec, 'Specificity (%)': spec,
    'Precision (%)': prec, 'F1-Score (%)': f1, 'AUC': auc, 'Error Rate (%)': 100-acc
}
pd.DataFrame([results]).to_csv('outputs/results.csv', index=False)
np.save('outputs/pso_weights.npy', best_weights)
print('Saved: outputs/results.csv  |  outputs/pso_weights.npy')

## 8. Grad-CAM Visualisation

Set `GRADCAM_IMG` to the path of a single MRI image to visualise.

In [ ]:
GRADCAM_IMG = None   # <-- Set to a path, e.g. '/data/tumor_001.jpg'

def make_gradcam_heatmap(img_array, model, last_conv_name, pred_index=None):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads   = tape.gradient(class_channel, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-9)
    return heatmap.numpy()

def last_conv_name(model):
    for layer in reversed(model.layers):
        if isinstance(layer, (layers.Conv2D, layers.DepthwiseConv2D)):
            return layer.name

if GRADCAM_IMG:
    img         = keras.preprocessing.image.load_img(GRADCAM_IMG, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr     = keras.preprocessing.image.img_to_array(img) / 255.0
    img_arr_exp = np.expand_dims(img_arr, 0)

    fig, axes = plt.subplots(2, N_MODELS, figsize=(3.5 * N_MODELS, 7))
    for col, (name, model) in enumerate(models.items()):
        heatmap = make_gradcam_heatmap(img_arr_exp, model, last_conv_name(model))
        heatmap_rgb = cm.jet(heatmap)[..., :3] * 255
        overlay = heatmap_rgb * 0.4 + img_arr * 255 * 0.6

        pred  = model.predict(img_arr_exp, verbose=0)
        label = CLASS_NAMES[np.argmax(pred)]

        axes[0][col].imshow(img); axes[0][col].set_title(name); axes[0][col].axis('off')
        axes[1][col].imshow(overlay.astype(np.uint8))
        axes[1][col].set_title(f'Grad-CAM\n{label}'); axes[1][col].axis('off')

    plt.suptitle('Grad-CAM Visualisations — All 6 Models', y=1.02, fontsize=13)
    plt.tight_layout(); plt.savefig('outputs/gradcam_all_models.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Set GRADCAM_IMG to a file path to view Grad-CAM heatmaps.')

---
## Summary

| Model | Role |
|---|---|
| MobileNet | Original (paper) |
| DenseNet121 | Original (paper) |
| MobileNetV2 | Original (paper) |
| Xception | Original (paper) |
| **InceptionV3** | **Added (new)** |
| **ResNet50** | **Added (new)** |

Each model is enhanced with a **ConvMixer block** and combined via a **PSO-optimised weighted-average ensemble** — the same approach as the original SwarmFusionDNN paper, now extended to 6 models.